# Base de connaissances d'un débat — propositions, arguments, conflits

**Notebook pédagogique CoursIA** · #1961 Phase 5 (assets CoursIA réutilisables) · `Claude Code @ myia-po-2025:2025-Epita-Intelligence-Symbolique`.

Corpus-free : tous les exemples sont synthétiques, domaine-public. Aucun LLM, aucune JVM — une base de connaissances de débat est un dictionnaire déterministe avec une convention de négation.

Ce que couvre ce notebook :

1. la **population transitive** : ajouter un argument enregistre ses prémisses et sa conclusion, sans appel explicite ;
2. la **recherche support/attaque** : qui conclut pour (ou contre) une proposition ;
3. la **convention de négation `¬`** : attaquer P = conclure `¬P` ; la cohérence = aucun couple P/`¬P` cohabitant ;
4. le **nommage honnête** : `entails` teste l'appartenance, pas une déduction logique — la docstring le dit, le notebook le vérifie.

**Garde round-trip** : chaque requête affichée ici est rejouée contre le moteur source par `tests/unit/coursia/knowledge_base/test_knowledge_base_roundtrip.py` (mêmes scénarios, même JSON partagé).


## §1 — Le modèle : population transitive

`add_argument` range l'argument **et** enregistre ses prémisses et sa conclusion comme propositions. Conséquence mesurable : `entails` répond `True` sur une prémisse que personne n'a ajoutée explicitement — c'est l'argument qui l'a portée.


In [1]:
# Imports + configuration. La base est déterministe (aucun LLM, aucune JVM).
import json
import logging
import os
import sys

logging.disable(logging.INFO)  # sorties propres malgré la chaîne d'import du dépôt

# Localiser la racine du dépôt (contient argumentation_analysis/) en remontant depuis le CWD.
_cwd = os.getcwd()
while _cwd and not os.path.isdir(os.path.join(_cwd, "argumentation_analysis")):
    _parent = os.path.dirname(_cwd)
    if _parent == _cwd:
        break
    _cwd = _parent
ROOT = _cwd if os.path.isdir(os.path.join(_cwd, "argumentation_analysis")) else os.getcwd()
sys.path.insert(0, ROOT)

from argumentation_analysis.agents.core.debate.knowledge_base import KnowledgeBase
from argumentation_analysis.agents.core.debate.protocols import FormalArgument, Proposition

EXAMPLES_PATH = os.path.join(
    ROOT, "docs", "coursia_contrib", "knowledge_base_examples.json"
)
with open(EXAMPLES_PATH, encoding="utf-8") as fh:
    examples = json.load(fh)


def construire(kb_spec):
    kb = KnowledgeBase()
    for arg_spec in kb_spec["arguments"]:
        kb.add_argument(
            FormalArgument(
                premises=[Proposition(content=p) for p in arg_spec["premises"]],
                conclusion=Proposition(content=arg_spec["conclusion"]),
                scheme=arg_spec["scheme"] or None,
            )
        )
    return kb


kbs = {spec["name"]: construire(spec) for spec in examples["kbs"]}
spec_deux_camps = next(s for s in examples["kbs"] if s["name"] == "deux_camps")

kb = kbs["deux_camps"]
print(f"Propositions enregistrées ({len(kb.get_all_propositions())}) :")
for p in kb.get_all_propositions():
    print(f"  - {p.content}")

premisse = Proposition(content="Le télétravail isole les collaborateurs")
assert kb.entails(premisse), "la prémisse doit être portée par l'argument"
print("\n'Le télétravail isole les collaborateurs' n'a jamais été ajoutée explicitement — "
      "entails répond True : l'argument l'a portée.")


Propositions enregistrées (4) :
  - Le télétravail isole les collaborateurs
  - Le télétravail réduit la concentration
  - Les pauses régulières restaurent la concentration
  - ¬Le télétravail réduit la concentration

'Le télétravail isole les collaborateurs' n'a jamais été ajoutée explicitement — entails répond True : l'argument l'a portée.


Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2026-09-14 18:50:48 [WARNING] [Services.CryptoService] crypto_service.__init__:46 - Service de chiffrement initialisé sans clé. Le chiffrement est désactivé.


## §2 — Support et attaque : qui conclut quoi

`find_supporting_arguments(P)` renvoie les arguments dont la **conclusion** est P ; `find_attacking_arguments(P)` ceux dont la conclusion est **`¬` + P**. La convention est lexicale : la négation est un préfixe de chaîne, pas un opérateur logique.


In [2]:
# Requêtes support/attaque rejouées depuis le JSON partagé, avec assertion.
prop = Proposition(content="Le télétravail réduit la concentration")

soutiens = kb.find_supporting_arguments(prop)
attaques = kb.find_attacking_arguments(prop)
print(f"Soutiens de « {prop.content} » : {len(soutiens)}")
for a in soutiens:
    print(f"  [{', '.join(str(p) for p in a.premises)}] -> {a.conclusion}")
print(f"Attaques (conclusion préfixée par ¬) : {len(attaques)}")
for a in attaques:
    print(f"  [{', '.join(str(p) for p in a.premises)}] -> {a.conclusion}")

assert len(soutiens) == 1 and len(attaques) == 1


Soutiens de « Le télétravail réduit la concentration » : 1
  [Le télétravail isole les collaborateurs] -> Le télétravail réduit la concentration
Attaques (conclusion préfixée par ¬) : 1
  [Les pauses régulières restaurent la concentration] -> ¬Le télétravail réduit la concentration


## §3 — Cohérence, et un nommage honnête

`is_consistent` répond `False` dès qu'une proposition **et** sa négation cohabitent — le débat sur P est ouvert. Et `entails` mérite qu'on le lise pour ce qu'il est : un test d'**appartenance** (`content in propositions`), pas un moteur d'inférence. La docstring du moteur le dit (« contains ») ; le nom est plus ambitieux que la sémantique — le savoir évite d'y brancher une attente déductive.


In [3]:
# Cohérence et appartenance, sur les deux bases du JSON partagé.
for spec in examples["kbs"]:
    k = kbs[spec["name"]]
    for q in spec["queries"]:
        if q["kind"] == "consistent":
            got = k.is_consistent()
            assert got == q["expected"], (spec["name"], q, got)
            print(f"{spec['name']:<13} is_consistent -> {got}   ({spec['description']})")
        elif q["kind"] == "entails":
            got = k.entails(Proposition(content=q["prop"]))
            assert got == q["expected"], (spec["name"], q, got)
            print(f"{spec['name']:<13} entails({q['prop'][:44]}…) -> {got}")


deux_camps    entails(Le télétravail isole les collaborateurs…) -> True
deux_camps    entails(Le télétravail augmente la productivité…) -> False
deux_camps    is_consistent -> False   (Deux arguments opposés sur la concentration en télétravail : la base contient une proposition ET sa négation.)
un_seul_camp  is_consistent -> True   (Un seul argument : aucune négation cohabitante, la base est cohérente.)


## Ce qu'il faut retenir

- `add_argument` est **transitif** : prémisses et conclusion entrent dans la base avec l'argument ;
- **support** = conclusion identique ; **attaque** = conclusion préfixée `¬` — une convention lexicale, simple et vérifiable ;
- `is_consistent` détecte le **conflit ouvert** (P et `¬P` cohabitants) — c'est lui qui signale qu'un débat a lieu ;
- `entails` est un test **d'appartenance** : nommage honnête dans la docstring, attente à ne pas surcharger.

Ce notebook est exécuté — les sorties ci-dessus sont réelles, produites par le moteur du dépôt, pas retouchées.

**Références** : moteur : `argumentation_analysis/agents/core/debate/knowledge_base.py` (adaptation du livrable étudiant `1_2_7_argumentation_dialogique`) · assets voisins : `argumentation_schemes.ipynb` (schémas), `dialogue_protocols.ipynb` (protocoles) · issue #1961 Phase 5 · garde : `tests/unit/coursia/knowledge_base/test_knowledge_base_roundtrip.py`.
